In [4]:
from trot.config import configure_once

configure_once()

In [5]:
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 8
U = 12
t = 1.0
n = L 
nelec =(L//2,L//2)
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = nelec
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)


System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Fri Oct  2 12:24:23 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:        20.54
[[[1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 

np.float64(-1.1597070476816338)

In [7]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
orbitals = mf.mo_coeff
nocc = L//2
trial_data = np.block([
    [orbitals[0][:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[1][:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.1597070477  2.000000e+02           0       0.0
[eql   20/100]   -1.2063800878  2.002611e+02           0       2.1
[eql   40/100]   -1.2320963894  2.003371e+02           0       4.1
[eql   60/100]   -1.2502892037  2.000307e+02           0       5.7
[eql   80/100]   -1.3159092230  2.001455e+02           0       7.3
[eql  100/100]   -1.6546258209  2.011148e+02           0       9.0

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.4652175939   1.353e-01     -1.4652175939  2.001350e+02           0      0.089      13.4
[blk  100/500]   -1.3814918158   7.475e-02     -1.2974411683  1.993614e+02           0      0.081      17.5
[blk  150/500]   -1.3531553495   5.311e-02     -1.2965301595  1.99916

KeyboardInterrupt: 

In [ ]:
# from trot.gmps.driver import make_mps_cpmc_ops
# from trot.gmps.utils import sd_to_gmps
# from trot.trial.mps import make_mps_trial
# from trot.prop.types import QmcParamsMps

# Ca, Cb = mf.mo_coeff[0][:, :L//2], mf.mo_coeff[1][:, :L//2]
# ghf = GhfTrial(jnp.array(np.block([[Ca, Cb], [Ca, -Cb]])))
# tensors = sd_to_gmps(Ca, Cb, mode="maximal").tensors
# R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
# M = np.zeros((4, 4))
# M[0, 0] = 1.0
# M[1:3, 1:3] = R
# M[3, 3] = -1
# new_tensors = [np.einsum("ps,asb->apb", M, np.asarray(A, dtype=float)) for A in tensors]
# trial = make_mps_trial(new_tensors,nelec=(L//2, L//2))
# params = QmcParamsMps(n_walkers=200, n_blocks=500, n_eql_blocks=100, seed=2,
#                       orbital_plan="maximal", walker_channel_chi=None)
# ops = make_mps_cpmc_ops(ham, trial, sys, params)
# run = run_qmc(sys=sys, params=params, ham_data=ham, trial_data=trial, trial_ops=ops.trial_ops,
#               meas_ops=ops.meas_ops, prop_ops=ops.prop_ops,
#               prop_ctx=ops.prop_ops.build_prop_ctx(ham, None, params), block_fn=blocks.block)


In [9]:
from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import (
    MpsTrial,
    _hashable_charges,
    _norm2,
    compress_mps_qn,
    one_rdm,
)

# STEP 1: Rotate the original MPS on each site's physical index.
# Local basis: empty, up, down, doubly occupied.
# This preserves total N, but mixes N_up and N_down.
Ca = mf.mo_coeff[0][:, :L // 2]
Cb = mf.mo_coeff[1][:, :L // 2]
tensors = sd_to_gmps(Ca, Cb, mode="maximal").tensors

R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
M = np.zeros((4, 4))
M[0, 0] = 1.0                  # Empty stays empty.
M[1:3, 1:3] = R                # Mix the two singly occupied states.
M[3, 3] = np.linalg.det(R)     # Fermionic sign for double occupancy.
new_tensors = [
    np.einsum("pq,aqb->apb", M, np.asarray(A, dtype=float))
    for A in tensors
]


physical_charges = np.array([
    [0, 0],   # empty
    [1, 0],   # up
    [0, 1],   # down
    [1, 1],   # doubly occupied
])
target_up, target_down = (int(value) for value in nelec)
print(target_down)
print(target_up)
num_sites = len(new_tensors)
# if not (0 <= target_up <= num_sites and 0 <= target_down <= num_sites):
#     raise ValueError("Each target particle count must lie between 0 and L.")

count_shape = (target_up + 1, target_down + 1)
print(count_shape)
# A new bond index will be a triple (old_index, up_so_far, down_so_far).
# Counts refer to sites LEFT of the bond, not to the whole chain.
# The same old index can occur with several different count pairs.

# For finding paths, only existence matters. Negative amplitudes are valid!
# Never sum signed amplitudes and test '> 0' to decide whether a path exists.
nonzero = [A != 0 for A in new_tensors]

# STEP 3: Walk FORWARD to find reachable bond states.
# reachable[s][a, u, d] is True if the first s sites can end at old
# bond index a with u up electrons and d down electrons.
reachable = [np.zeros((1,) + count_shape, dtype=bool)]
reachable[0][0, 0, 0] = True    # Before the first site: no electrons.

for site, A in enumerate(new_tensors):
    #Possible states reachable after 
    next_reachable = np.zeros((A.shape[2],) + count_shape, dtype=bool)
    #Looping over the coordinates where the site is reachable
    for old_left, up, down in np.argwhere(reachable[site]):
        #For each reachable triple, try all four possible local states:
        for physical, (add_up, add_down) in enumerate(physical_charges):
            next_up = up + add_up
            next_down = down + add_down
            #Skip any choice that already exceeds a target count. This is safe because processing more sites only adds electrons.
            if next_up > target_up or next_down > target_down:
                #advances to the next physical state.
                continue      

            #Fix the original left index and physical state, then inspect all possible original right indices.
            right_indices = np.flatnonzero(nonzero[site][old_left, physical, :])
            next_reachable[right_indices, next_up, next_down] = True

    reachable.append(next_reachable)

if not reachable[-1][0, target_up, target_down]:
    raise ValueError("No tensor path reaches the requested particle sector.")

# STEP 4: Walk BACKWARD to remove paths that cannot finish at the target.
# alive[s][a, u, d] means both:
#   (1) reachable from the left boundary, and
#   (2) able to reach the target at the right boundary.
alive = [np.zeros_like(bond_reachable) for bond_reachable in reachable]
alive[-1][0, target_up, target_down] = True

for site in range(num_sites - 1, -1, -1):
    for old_left, up, down in np.argwhere(reachable[site]):
        for physical, (add_up, add_down) in enumerate(physical_charges):
            next_up = up + add_up
            next_down = down + add_down
            if next_up > target_up or next_down > target_down:
                continue

            right_indices = np.flatnonzero(nonzero[site][old_left, physical, :])
            can_finish = np.any(alive[site + 1][right_indices, next_up, next_down])
            if can_finish:
                alive[site][old_left, up, down] = True
                break         # One valid continuation is enough.

# STEP 5: Give each surviving triple its own new bond index.
# For example, (a, 1, 0) and (a, 0, 1) are DIFFERENT new indices.
# Sort by (up, down, old_index) so equal charge labels sit together.
states = []
for bond_alive in alive:
    triples = [tuple(row) for row in np.argwhere(bond_alive)]
    triples.sort(key=lambda row: (row[1], row[2], row[0]))
    states.append(np.asarray(triples, dtype=int).reshape(-1, 3))

# STEP 6: Copy the amplitudes into tensors with the enlarged bond indices.
# Keep a connection only when:
#   right_counts = left_counts + physical_charge.
# Starting at (0, 0) and ending at the target then enforces the projection.
# Every configuration in the target sector keeps its original amplitude.
projected = []
for site, A in enumerate(new_tensors):
    left_states = states[site]
    right_states = states[site + 1]
    right_lookup = {tuple(state): index for index, state in enumerate(right_states)}
    B = np.zeros((len(left_states), 4, len(right_states)), dtype=A.dtype)

    for new_left, (old_left, up, down) in enumerate(left_states):
        for physical, (add_up, add_down) in enumerate(physical_charges):
            for old_right in np.flatnonzero(nonzero[site][old_left, physical, :]):
                right_state = (old_right, up + add_up, down + add_down)
                new_right = right_lookup.get(right_state)
                if new_right is not None:
                    # Copy the actual signed amplitude, not the nonzero mask.
                    B[new_left, physical, new_right] = A[old_left, physical, old_right]

    projected.append(B)

# STEP 7: Measure the retained weight BEFORE normalization or compression.
# The Boolean passes find paths, but cannot detect amplitude cancellations.
# A reachable sector may therefore still have zero (or negligible) norm.
norm_total = _norm2(new_tensors)
sector_norm2 = _norm2(projected)
if not sector_norm2 > 1.0e-12 * norm_total:
    raise ValueError("The requested sector has zero or negligible weight.")
sector_weight = float(sector_norm2 / norm_total)

# Each new index has a definite pair (up_so_far, down_so_far).
labels = tuple(bond_states[:, 1:].copy() for bond_states in states)

# STEP 8: Remove redundant bond directions WITHOUT mixing charge labels.
# QR/SVD run separately in each charge sector. The default relative
# singular-value cutoff is 1e-13: the result is exact up to that truncation.
projected, labels = compress_mps_qn(projected, labels)
norm2 = _norm2(projected)

# Scaling one tensor scales the whole MPS. Normalize the projected state.
projected[0] = projected[0] / np.sqrt(norm2)

# STEP 9: Package the normalized tensors for CPMC.
# Match make_mps_trial's default: rdm1 comes from the ROTATED, unprojected
# state. It is auxiliary trial data, not the projected state's density.
rdm1 = np.stack(one_rdm(new_tensors))
trial = MpsTrial(
    tensors=tuple(jnp.asarray(A) for A in projected),
    rdm1=jnp.asarray(rdm1),
    charges=_hashable_charges(labels),
    nelec=(target_up, target_down),
    sector_weight=sector_weight,
)

print(f"Target: (N_up, N_down) = {(target_up, target_down)}")
print(f"S_z = {(target_up - target_down) / 2:g}")
print(f"Fraction of rotated norm retained: {sector_weight:.8f}")
print(f"Normalized projected norm squared: {_norm2(projected):.12f}")


4
4
(5, 5)
[[[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]]

 [[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]]

 [[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]]

 [[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]]]
[[[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False False]]

 [[False False False False False]
  [False False False False False]
  [False False False False False]
  [False False False False 

In [11]:
from trot.meas.mps import apply_mpo, hubbard_mpo_from_h1
from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import make_mps_trial, rotate_spin


def contract(bra, ket):
    """<bra|ket> of two real d = 4 MPS in the same local basis."""
    env = np.ones((1, 1))
    for x, y in zip(bra, ket):
        env = np.einsum("ab,apc,bpd->cd", env, np.asarray(x), np.asarray(y), optimize=True)
    return float(env[0, 0])

nelec = (L // 2, L // 2)
Ca, Cb = mf.mo_coeff[0][:, :L // 2], mf.mo_coeff[1][:, :L // 2]
R = np.array([[1.0, 1.0], [1.0, -1.0]]) / np.sqrt(2.0)
M = np.zeros((4, 4))
M[0, 0], M[1:3, 1:3], M[3, 3] = 1.0, R, -1.0

gmps = sd_to_gmps(Ca, Cb, mode="maximal")  
way_b = rotate_spin(gmps.tensors, R)  # the single-site rotation only

labels_b = [np.asarray(q).reshape(-1, 2).sum(axis=1) for q in gmps.charges]  # the rotation keeps N on every index
W = hubbard_mpo_from_h1(h1, U)
h_b = apply_mpo(W, way_b)

rng = np.random.default_rng(0)
for k in range(6):  # random S_z = 0 determinants as exact MPS in the same local basis
    Wa = np.linalg.qr(rng.standard_normal((L, nelec[0])))[0]
    Wb = np.linalg.qr(rng.standard_normal((L, nelec[1])))[0]
    phi = sd_to_gmps(Wa, Wb, mode="maximal").tensors
    o_b = contract(way_b, phi)

